# GAT Basic — Next-Event Prediction on BPI Challenge 2012

This notebook demonstrates the **basic Dual-GAT** model on the
[BPI Challenge 2012](https://doi.org/10.4121/uuid:3926db30-f712-4394-aebc-75976070e91f) dataset,
a large-scale loan application process log.

## Dataset summary

| Property | Value |
|----------|-------|
| Domain | Financial loan applications |
| Cases | ~13,000 |
| Events | ~262,000 |
| Activities | 36 |
| Lifecycle statuses | COMPLETE, SCHEDULE, START, … |
| Numeric feature | `AMOUNT_REQ` (requested loan amount) |

Compared to Helpdesk this is a much larger and richer log — ideal for benchmarking.

### Notebook outline
1. Load & Explore
2. Train
3. Evaluate
4. Summary

In [1]:
from pathlib import Path
import sys, warnings

root = Path.cwd().resolve()
src = root / "src" if (root / "src").exists() else root.parent / "src"
sys.path.insert(0, str(src))
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Load & Explore the Dataset

In [2]:
import pandas as pd

data_dir = Path.cwd() / "data" if (Path.cwd() / "data").exists() else Path.cwd() / "examples" / "data"
df = pd.read_csv(data_dir / "BPI_Challenge_2012.csv")

df = df.rename(columns={
    "case:concept:name": "case_id",
    "concept:name":      "event",
    "time:timestamp":    "time",
    "lifecycle:transition": "status",
    "case:AMOUNT_REQ":      "amount_req",
})
df = df.dropna(subset=["case_id", "event", "time"]).copy()
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
df = df[df.groupby("case_id")["case_id"].transform("size") >= 2].reset_index(drop=True)

n_cases  = df["case_id"].nunique()
n_events = len(df)
n_acts   = df["event"].nunique()
avg_len  = df.groupby("case_id").size().mean()

print(f"Cases: {n_cases}  |  Events: {n_events}  |  Activities: {n_acts}  |  Avg length: {avg_len:.1f}")
df.head()

Cases: 13087  |  Events: 262200  |  Activities: 24  |  Avg length: 20.0


,org:resource,status,event,time,case:REG_DATE,case_id,amount_req
0,112.0,COMPLETE,A_SUBMITTED,2011-10-01 00:38:44.546000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
1,112.0,COMPLETE,A_PARTLYSUBMITTED,2011-10-01 00:38:44.880000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
2,112.0,COMPLETE,A_PREACCEPTED,2011-10-01 00:39:37.906000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
3,112.0,SCHEDULE,W_Completeren aanvraag,2011-10-01 00:39:38.875000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
4,NaN,START,W_Completeren aanvraag,2011-10-01 11:36:46.437000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000


## 2. Train the Model

We use `amount_req` (loan amount) as a numeric sequence-level feature.
Since BPI 2012 is large, training may take several minutes per epoch.

In [3]:
from timegnn import gat_basic

result = gat_basic(
    df,
    case_col="case_id",
    event_col="event",
    time_col="time",
    cat_event=["event"],
    num_event=[],
    seq_cols=["amount_req"],
    cat_seq=[],
    num_seq=["amount_req"],
    num_epochs=3,
    batch_size=16,
)

model = result["model"]
pd.DataFrame(result["history"])

,epoch,train_loss,train_acc,test_loss,test_acc
0,1,0.563658,0.805846,0.549832,0.817428
1,2,0.496067,0.819361,0.544005,0.796309
2,3,0.490391,0.821481,0.500074,0.834262


## 3. Evaluate with Comprehensive Metrics

In [4]:
import torch
from torch.utils.data import DataLoader
from timegnn import (
    EventLogTransformer, top_k_accuracy, predict, predict_per_sequence,
    average_bleu_score, compute_dls_and_exact_match, analyze_sequence_errors,
)
from timegnn.data.pyg import CustomDataset, custom_collate_fn

transformer = EventLogTransformer(
    case_col="case_id", event_col="event", time_col="time",
    cat_event=["event"], num_event=[],
    seq_cols=["amount_req"], cat_seq=[], num_seq=["amount_req"],
    mode="gat_basic",
).fit(df)

transformed = transformer.transform(df)
dataset = CustomDataset(transformed.event_features, transformed.labels)
loader  = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=custom_collate_fn)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)

preds_flat, labels_flat, outputs = predict(model, loader, device)
print(f"Top-1 accuracy: {top_k_accuracy(outputs, labels_flat, k=1):.4f}")
print(f"Top-3 accuracy: {top_k_accuracy(outputs, labels_flat, k=3):.4f}")
print(f"Top-5 accuracy: {top_k_accuracy(outputs, labels_flat, k=5):.4f}")

preds_seq, labels_seq = predict_per_sequence(model, loader, device)
bleu = average_bleu_score(preds_seq, labels_seq)
dls, exact = compute_dls_and_exact_match(preds_seq, labels_seq)
print(f"\nBLEU:  {bleu:.4f}")
print(f"DLS:   {dls:.4f}")
print(f"Exact: {exact:.4f}")

print("\nSequence error summary:")
print(analyze_sequence_errors(model, loader, device))

Top-1 accuracy: 0.8293
Top-3 accuracy: 0.9786
Top-5 accuracy: 0.9933

BLEU:  0.5226
DLS:   0.7974
Exact: 0.1007

Sequence error summary:
(array([   0, 7885,    0, 2559, 2061, 1369, 1062, 1344,  786,  758,  471,
        495,  815,  355,  879,  411,  939,  713,  895,  740,  952,  792,
        837,  822,  825,  848,  703,  741,  682,  711,  607,  651,  571,
        592,  580,  546,  520,  476,  449,  447,  390,  397,  370,  348,
        298,  318,  295,  274,  265,  256,  226,  217,  192,  191,  156,
        153,  130,  133,  131,  113,  121,  110,   94,   96,  109,  105,
         82,   81,   73,   69,   59,   62,   64,   56,   47,   50,   45,
         47,   44,   42,   29,   34,   32,   21,   18,   21,   15,   19,
         20,   12,   15,   22,   11,   16,   14,   18,   11,   14,   16,
          6,    8,   11,    7,    9,    8,   15,    9,    8,    9,    7,
          7,    5,    6,    6,    6,    4,    4,    2,    2,    4,    3,
          5,    4,    4,    4,    0,    3,    2,    3,    1

## 4. Summary

BPI 2012 is significantly larger than Helpdesk, so training takes longer
but the model benefits from more data.

To get better results:
- Increase `num_epochs` (10–20)
- Use GPU acceleration (`device="cuda"`)
- Try richer model variants: `gat_status`, `gat_time_decay`, `gat_time_decay_status`